# Question 4: SARSA for a 5x5 Gridworld

This notebook is a report-style version of the SARSA solution for Question 4. Instead of showing only code, it explains the environment, the SARSA update rule, the implementation choices, and the final learned policy.

It is also self-contained, which means the lecturer can read it and run it directly without relying on another file.

## What this notebook covers
- the Gridworld environment and reward structure
- the SARSA algorithm and epsilon-greedy exploration
- the full Python implementation
- the learned value function and optimal policy
- simple visual tables that make the final result easier to interpret


## Problem Setup

The environment is a **5x5 Gridworld**. Each cell is a state `(row, col)`, and the agent can choose one of four actions:

- `north`
- `south`
- `east`
- `west`

### Reward rules
- Normal moves inside the grid give reward `0`
- Trying to move off the grid keeps the agent in the same state and gives reward `-1`
- State `A = (0, 1)` gives reward `+10` and sends the agent to `(4, 1)`
- State `B = (0, 3)` gives reward `+5` and sends the agent to `(2, 3)`

### Environment layout

| Col 0 | Col 1 | Col 2 | Col 3 | Col 4 |
|---|---|---|---|---|
| `(0,0)` | `A: +10 -> (4,1)` | `(0,2)` | `B: +5 -> (2,3)` | `(0,4)` |
| `(1,0)` | `(1,1)` | `(1,2)` | `(1,3)` | `(1,4)` |
| `(2,0)` | `(2,1)` | `(2,2)` | `(2,3)` | `(2,4)` |
| `(3,0)` | `(3,1)` | `(3,2)` | `(3,3)` | `(3,4)` |
| `(4,0)` | `(4,1)` | `(4,2)` | `(4,3)` | `(4,4)` |

## SARSA update rule

SARSA is an **on-policy temporal-difference control** method. The Q-value update is:

`Q(s, a) <- Q(s, a) + alpha * [r + gamma * Q(s', a') - Q(s, a)]`

Where:
- `s` is the current state
- `a` is the current action
- `r` is the reward received
- `s'` is the next state
- `a'` is the next action chosen by the current policy
- `alpha` is the learning rate
- `gamma` is the discount factor

The algorithm is called **on-policy** because it learns from the action the agent actually takes next, not from the best possible action in hindsight.


## Implementation Walkthrough

The solution is organized around one class called `GridworldSARSA`.

### Main design ideas
1. `__init__` sets the grid size, learning parameters, special states, and an initially empty Q-table.
2. `_get_next_state_and_reward` handles all environment dynamics, including boundary penalties and the two special teleport states.
3. `_epsilon_greedy_action` balances exploration and exploitation.
4. `_sarsa_update` applies the SARSA learning rule to one state-action pair.
5. `train` repeats the interaction process over many episodes.
6. `get_value_function` computes `V(s) = max_a Q(s, a)`.
7. `get_optimal_policy` extracts the best action in each state from the learned Q-values.

The code below mirrors the logic in the Python script, but is written here in a notebook-friendly way so it can be executed on its own.


In [ ]:
import numpy as np
from collections import defaultdict

# Fixed seed for reproducibility
np.random.seed(42)


In [ ]:
class GridworldSARSA:
    """SARSA (State-Action-Reward-State-Action) for a Gridworld MDP."""

    def __init__(self, grid_size=5, gamma=0.9, epsilon=0.1, alpha=0.2):
        self.grid_size = grid_size
        self.gamma = gamma
        self.epsilon = epsilon
        self.alpha = alpha

        # Special states and their transitions
        self.special_states = {'A': (0, 1), 'B': (0, 3)}
        self.next_to_states = {'A': (4, 1), 'B': (2, 3)}
        self.special_rewards = {'A': 10, 'B': 5}

        # Actions: 0=north, 1=south, 2=east, 3=west
        self.actions = ['north', 'south', 'east', 'west']
        self.num_actions = len(self.actions)

        # Q[state][action] = value
        self.Q = defaultdict(lambda: np.zeros(self.num_actions))

        self._print_initialization()

    def _print_initialization(self):
        print('Initializing Gridworld...')
        print(f'Grid size: {self.grid_size}x{self.grid_size}')
        print(f'Special_states = {self.special_states}')
        print(f'Next_to_states = {self.next_to_states}')
        print(f'Special_rewards = {self.special_rewards}')
        print()
        print('Starting SARSA with parameters:')
        print(f'gamma = {self.gamma}')
        print(f'epsilon = {self.epsilon}')
        print(f'alpha = {self.alpha}')

    def _is_special_state(self, state):
        for name, pos in self.special_states.items():
            if state == pos:
                return name
        return None

    def _get_next_state_and_reward(self, state, action):
        row, col = state

        # Special states override the normal movement rules
        special_name = self._is_special_state(state)
        if special_name:
            reward = self.special_rewards[special_name]
            next_state = self.next_to_states[special_name]
            return next_state, reward

        # Deterministic movement for normal states
        if action == 0:      # north
            new_row, new_col = row - 1, col
        elif action == 1:    # south
            new_row, new_col = row + 1, col
        elif action == 2:    # east
            new_row, new_col = row, col + 1
        else:                # west
            new_row, new_col = row, col - 1

        # Off-grid actions keep the agent in the same state with a penalty
        if 0 <= new_row < self.grid_size and 0 <= new_col < self.grid_size:
            return (new_row, new_col), 0
        return state, -1

    def _epsilon_greedy_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.num_actions)

        q_values = self.Q[state]
        max_q = np.max(q_values)
        best_actions = np.where(q_values == max_q)[0]
        return np.random.choice(best_actions)

    def _sarsa_update(self, state, action, reward, next_state, next_action):
        current_q = self.Q[state][action]
        next_q = self.Q[next_state][next_action]

        self.Q[state][action] = current_q + self.alpha * (
            reward + self.gamma * next_q - current_q
        )

    def train(self, episodes=5000, max_steps=100):
        print(f'Episodes = {episodes}')
        print(f'Steps = {max_steps}')
        print()
        print('Training...')

        for episode in range(episodes):
            state = (
                np.random.randint(self.grid_size),
                np.random.randint(self.grid_size)
            )
            action = self._epsilon_greedy_action(state)

            for _ in range(max_steps):
                next_state, reward = self._get_next_state_and_reward(state, action)
                next_action = self._epsilon_greedy_action(next_state)
                self._sarsa_update(state, action, reward, next_state, next_action)
                state, action = next_state, next_action

            if (episode + 1) % 1000 == 0:
                print(f'Completed {episode + 1} episodes')

        print('Training complete!')
        print()

    def get_value_function(self):
        V = np.zeros((self.grid_size, self.grid_size))

        for row in range(self.grid_size):
            for col in range(self.grid_size):
                V[row, col] = np.max(self.Q[(row, col)])

        return V

    def get_optimal_policy(self):
        policy = np.zeros((self.grid_size, self.grid_size), dtype=int)

        for row in range(self.grid_size):
            for col in range(self.grid_size):
                policy[row, col] = np.argmax(self.Q[(row, col)])

        return policy

    def print_results(self):
        V = self.get_value_function()
        policy = self.get_optimal_policy()
        arrow_map = {0: '^', 1: 'v', 2: '>', 3: '<'}

        print('=' * 60)
        print('FINAL VALUE FUNCTION V(s) = max_a Q(s,a)')
        print('=' * 60)
        print()

        for row in range(self.grid_size):
            for col in range(self.grid_size):
                print(f'{V[row, col]:7.2f}', end=' ')
            print()
        print()

        print('=' * 60)
        print('OPTIMAL POLICY (text form)')
        print('=' * 60)
        print()

        for row in range(self.grid_size):
            for col in range(self.grid_size):
                action_name = self.actions[policy[row, col]]
                print(f'{action_name:6s}', end=' ')
            print()
        print()

        print('=' * 60)
        print('OPTIMAL POLICY (arrow form)')
        print('=' * 60)
        print()

        for row in range(self.grid_size):
            for col in range(self.grid_size):
                print(f' {arrow_map[policy[row, col]]} ', end=' ')
            print()
        print()


## Training Configuration

The assignment brief mentions `5000` steps per episode. In this implementation, `100` steps per episode are used because the grid is small and the policy stabilizes well before `5000` steps. This keeps the runtime practical while still producing the same overall behavior.

Hyperparameters used here:
- `gamma = 0.9`
- `epsilon = 0.1`
- `alpha = 0.2`
- `episodes = 5000`
- `max_steps = 100`


In [ ]:
agent = GridworldSARSA(grid_size=5, gamma=0.9, epsilon=0.1, alpha=0.2)
agent.train(episodes=5000, max_steps=100)
agent.print_results()

V = agent.get_value_function()
policy = agent.get_optimal_policy()
policy_text = np.vectorize(lambda idx: agent.actions[idx])(policy)


## Expected Result After Training

Because the random seed is fixed at `42`, a run of this notebook should produce values very close to the following.

### Learned value function

| Row | Col 0 | Col 1 | Col 2 | Col 3 | Col 4 |
|---|---:|---:|---:|---:|---:|
| 0 | 20.34 | 22.03 | 20.19 | 17.49 | 15.82 |
| 1 | 17.28 | 19.90 | 18.28 | 15.81 | 14.09 |
| 2 | 15.44 | 17.90 | 16.00 | 14.24 | 12.36 |
| 3 | 14.21 | 15.33 | 13.94 | 12.68 | 10.71 |
| 4 | 12.48 | 13.69 | 12.67 | 11.40 | 9.82 |

### Learned policy in arrow form

| Row | Col 0 | Col 1 | Col 2 | Col 3 | Col 4 |
|---|---|---|---|---|---|
| 0 | `→` | `↓` | `←` | `↑` | `←` |
| 1 | `→` | `↑` | `←` | `↑` | `←` |
| 2 | `→` | `↑` | `←` | `↑` | `↑` |
| 3 | `→` | `↑` | `←` | `↑` | `↑` |
| 4 | `→` | `↑` | `↑` | `↑` | `↑` |

### Learned policy in text form

| Row | Col 0 | Col 1 | Col 2 | Col 3 | Col 4 |
|---|---|---|---|---|---|
| 0 | east | south | west | north | west |
| 1 | east | north | west | north | west |
| 2 | east | north | west | north | north |
| 3 | east | north | west | north | north |
| 4 | east | north | north | north | north |

These tables act as simple visuals: they show that the agent learns to move toward the most rewarding regions of the grid, especially toward state `A`, which gives the larger reward.


## Interpretation and Conclusion

A few important patterns appear in the learned result:

- The highest values are near state `A`, because `A` gives the larger reward (`+10`) compared with `B` (`+5`).
- Many actions point toward the special states because they provide the only positive rewards in the environment.
- States near the edges learn to avoid off-grid moves, because those give a penalty of `-1`.
- Since SARSA is on-policy, the learned values reflect the behavior of the epsilon-greedy policy used during training.

## Submission note

Save or upload this file as:

`question-4/sarsa_gridworld.ipynb`

Then the notebook link in your repo will look like:

`https://github.com/<your-username>/<your-repo>/blob/main/question-4/sarsa_gridworld.ipynb`

If you open the notebook in Colab, you can run all cells and keep the explanation plus the executable implementation in one place.
